In [2]:
import pandas as pd

## cleaning data of nav_history.csv


In [3]:
import pandas as pd
from pathlib import Path

# Load file
df = pd.read_csv("02_nav_history.csv")

# --------------------------------------------------
# 1. Parse dates
# --------------------------------------------------
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce",
    dayfirst=True
)

# Remove invalid dates
df = df.dropna(subset=["date"])

# --------------------------------------------------
# 2. Remove duplicates
# Keep latest record if duplicates exist
# --------------------------------------------------
df = df.drop_duplicates(
    subset=["amfi_code", "date"],
    keep="last"
)

# --------------------------------------------------
# 3. Validate NAV values
# --------------------------------------------------
df["nav"] = pd.to_numeric(df["nav"], errors="coerce")

invalid_nav = df[df["nav"].isna() | (df["nav"] <= 0)]

print(f"Invalid NAV records: {len(invalid_nav)}")

# Remove invalid NAV rows
df = df[df["nav"] > 0]

# --------------------------------------------------
# 4. Sort by AMFI code and date
# --------------------------------------------------
df = df.sort_values(
    by=["amfi_code", "date"]
)

# --------------------------------------------------
# 5. Create continuous daily calendar
#    and forward-fill NAV for holidays/weekends
# --------------------------------------------------
cleaned_data = []

for amfi_code, group in df.groupby("amfi_code"):

    group = group.set_index("date")

    full_dates = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="D"
    )

    group = group.reindex(full_dates)

    group["amfi_code"] = amfi_code

    # Forward fill NAV
    group["nav"] = group["nav"].ffill()

    group = group.reset_index()
    group.rename(columns={"index": "date"}, inplace=True)

    cleaned_data.append(group)

df_clean = pd.concat(cleaned_data, ignore_index=True)

# --------------------------------------------------
# 6. Final validation
# --------------------------------------------------
assert (df_clean["nav"] > 0).all(), \
    "Found NAV <= 0 after cleaning"

# --------------------------------------------------
# 7. Save cleaned file
# --------------------------------------------------
# Create directory if it doesn't exist
Path("Data/processed").mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    "nav_history_clean.csv",
    index=False
)

print("Cleaning completed.")
print(df_clean.head())
print(df_clean.info())

FileNotFoundError: [Errno 2] No such file or directory: '02_nav_history.csv'

In [ ]:
import os
print(os.path.exists("Data/raw/02_nav_history.csv"))

False


In [ ]:
output_file = "Data/processed/nav_history_clean.csv"

df_clean.to_csv(output_file, index=False)

print(f"Saved successfully: {output_file}")
print("File exists:", Path(output_file).exists())

Saved successfully: Data/processed/nav_history_clean.csv
File exists: True


In [4]:
import pandas as pd
from pathlib import Path

In [5]:
# Load file
df = pd.read_csv("08_investor_transactions.csv")

# --------------------------------------------------
# 1. Fix date formats
# --------------------------------------------------
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce",
    dayfirst=True
)

invalid_dates = df["transaction_date"].isna().sum()
print(f"Invalid dates: {invalid_dates}")

df = df.dropna(subset=["transaction_date"])

# --------------------------------------------------
# 2. Standardise transaction_type
# --------------------------------------------------
df["transaction_type"] = (
    df["transaction_type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

mapping = {
    "SIP": "SIP",
    "SYSTEMATIC INVESTMENT PLAN": "SIP",

    "LUMPSUM": "Lumpsum",
    "LUMP SUM": "Lumpsum",

    "REDEMPTION": "Redemption",
    "REDEEM": "Redemption",
    "WITHDRAWAL": "Redemption"
}

df["transaction_type"] = df["transaction_type"].replace(mapping)

valid_transaction_types = ["SIP", "Lumpsum", "Redemption"]

invalid_txn = df[
    ~df["transaction_type"].isin(valid_transaction_types)
]

print(f"Invalid transaction types: {len(invalid_txn)}")

# --------------------------------------------------
# 3. Validate amount > 0
# --------------------------------------------------
df["amount_inr"] = pd.to_numeric(
    df["amount_inr"],
    errors="coerce"
)

invalid_amounts = df[
    df["amount_inr"].isna() |
    (df["amount_inr"] <= 0)
]

print(f"Invalid amounts: {len(invalid_amounts)}")

df = df[df["amount_inr"] > 0]

# --------------------------------------------------
# 4. Standardise KYC status
# --------------------------------------------------
df["kyc_status"] = (
    df["kyc_status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

kyc_mapping = {
    "Y": "VERIFIED",
    "YES": "VERIFIED",
    "VERIFIED": "VERIFIED",
    "COMPLETE": "VERIFIED",

    "N": "PENDING",
    "NO": "PENDING",
    "PENDING": "PENDING",
    "INCOMPLETE": "PENDING",

    "REJECTED": "REJECTED"
}

df["kyc_status"] = df["kyc_status"].replace(kyc_mapping)

valid_kyc = [
    "VERIFIED",
    "PENDING",
    "REJECTED"
]

invalid_kyc = df[
    ~df["kyc_status"].isin(valid_kyc)
]

print(f"Invalid KYC status records: {len(invalid_kyc)}")

# --------------------------------------------------
# 5. Remove duplicates
# --------------------------------------------------
df = df.drop_duplicates()

# --------------------------------------------------
# 6. Sort records
# --------------------------------------------------
df = df.sort_values(
    by=["investor_id", "transaction_date"]
)

# --------------------------------------------------
# 7. Save cleaned file
# --------------------------------------------------
Path("Data/processed").mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    "Data/processed/investor_transactions_clean.csv",
    index=False
)

# --------------------------------------------------
# 8. Summary
# --------------------------------------------------
print("\nCleaning Summary")
print("------------------")
print("Rows:", len(df))
print("Transaction Types:")
print(df["transaction_type"].value_counts())

print("\nKYC Status:")
print(df["kyc_status"].value_counts())

print("\nFile saved successfully.")

Invalid dates: 0
Invalid transaction types: 0
Invalid amounts: 0
Invalid KYC status records: 0

Cleaning Summary
------------------
Rows: 32778
Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
kyc_status
VERIFIED    30146
PENDING      2632
Name: count, dtype: int64

File saved successfully.


In [6]:
print(df.isnull().sum())

print(
    df["transaction_type"]
    .value_counts(dropna=False)
)

print(
    df["kyc_status"]
    .value_counts(dropna=False)
)

print(
    (df["amount_inr"] <= 0).sum()
)

investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64
kyc_status
VERIFIED    30146
PENDING      2632
Name: count, dtype: int64
0


In [7]:


# Load file
df = pd.read_csv("07_scheme_performance.csv")

# --------------------------------------------------
# 1. Numeric conversion
# --------------------------------------------------

numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct",
    "morningstar_rating"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --------------------------------------------------
# 2. Find non-numeric records
# --------------------------------------------------

invalid_numeric = df[df[numeric_cols].isna().any(axis=1)]

print("Rows with numeric conversion issues:",
      len(invalid_numeric))

# --------------------------------------------------
# 3. Expense Ratio Validation
# --------------------------------------------------

expense_anomalies = df[
    (df["expense_ratio_pct"] < 0.1) |
    (df["expense_ratio_pct"] > 2.5)
]

print("Expense ratio anomalies:",
      len(expense_anomalies))

# --------------------------------------------------
# 4. Return Anomaly Checks
# --------------------------------------------------

return_anomalies = df[
    (df["return_1yr_pct"] < -100) |
    (df["return_1yr_pct"] > 100) |

    (df["return_3yr_pct"] < -100) |
    (df["return_3yr_pct"] > 100) |

    (df["return_5yr_pct"] < -100) |
    (df["return_5yr_pct"] > 100)
]

print("Return anomalies:",
      len(return_anomalies))

# --------------------------------------------------
# 5. Beta Validation
# --------------------------------------------------

beta_anomalies = df[
    (df["beta"] < 0) |
    (df["beta"] > 3)
]

print("Beta anomalies:",
      len(beta_anomalies))

# --------------------------------------------------
# 6. Morningstar Rating Validation
# --------------------------------------------------

rating_anomalies = df[
    ~df["morningstar_rating"].isin(
        [1, 2, 3, 4, 5]
    )
]

print("Rating anomalies:",
      len(rating_anomalies))

# --------------------------------------------------
# 7. AUM Validation
# --------------------------------------------------

aum_anomalies = df[
    df["aum_crore"] <= 0
]

print("AUM anomalies:",
      len(aum_anomalies))

# --------------------------------------------------
# 8. Consolidated anomaly file
# --------------------------------------------------

all_anomalies = pd.concat([
    invalid_numeric,
    expense_anomalies,
    return_anomalies,
    beta_anomalies,
    rating_anomalies,
    aum_anomalies
]).drop_duplicates()

# --------------------------------------------------
# 9. Clean dataset
# --------------------------------------------------

df_clean = df.copy()

df_clean = df_clean[
    (df_clean["expense_ratio_pct"] >= 0.1) &
    (df_clean["expense_ratio_pct"] <= 2.5)
]

df_clean = df_clean[
    df_clean["aum_crore"] > 0
]

# --------------------------------------------------
# 10. Save files
# --------------------------------------------------

Path("Data/processed").mkdir(
    parents=True,
    exist_ok=True
)

df_clean.to_csv(
    "Data/processed/scheme_performance_clean.csv",
    index=False
)

all_anomalies.to_csv(
    "Data/processed/scheme_performance_anomalies.csv",
    index=False
)

# --------------------------------------------------
# 11. Summary
# --------------------------------------------------

print("\nCleaning Summary")
print("------------------")
print("Original Rows:", len(df))
print("Clean Rows:", len(df_clean))
print("Anomaly Rows:", len(all_anomalies))

Rows with numeric conversion issues: 0
Expense ratio anomalies: 0
Return anomalies: 0
Beta anomalies: 0
Rating anomalies: 0
AUM anomalies: 0

Cleaning Summary
------------------
Original Rows: 40
Clean Rows: 40
Anomaly Rows: 0


In [4]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\raw\\01_fund_master.csv")

df["launch_date"] = pd.to_datetime(
    df["launch_date"],
    errors="coerce"
)

df["expense_ratio_pct"] = pd.to_numeric(
    df["expense_ratio_pct"],
    errors="coerce"
)

df = df.drop_duplicates(
    subset=["amfi_code"]
)

C:\Users\Mohit\AppData\Local\Temp\ipykernel_22412\3055775096.py:5: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["launch_date"] = pd.to_datetime(


In [5]:
df.to_csv("data/processed/fund_master_clean.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\raw\\03_aum_by_fund_house.csv")

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

numeric_cols = [
    "aum_lakh_crore",
    "num_schemes",
    "aum_crore"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df[df["aum_crore"] > 0]



C:\Users\Mohit\AppData\Local\Temp\ipykernel_22412\4008746324.py:5: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(


OSError: Cannot save file into a non-existent directory: 'data\processed'

In [10]:
df.to_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\processed\\aum_by_fund_house_clean.csv", index=False)

## cleaning monthly slip number 4


In [3]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\raw\\04_monthly_sip_inflows.csv")

df["month"] = pd.to_datetime(
    df["month"],
    errors="coerce"
)

for col in [
    "sip_inflow_crore",
    "active_sip_accounts_crore",
    "new_sip_accounts_lakh",
    "sip_aum_lakh_crore",
    "yoy_growth_pct"
]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [4]:
df.to_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\processed\\monthly_sip_inflows_lean.csv", index=False)

### category inflows no- 5

In [6]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\raw\\05_category_inflows.csv")

df["month"] = pd.to_datetime(
    df["month"],
    errors="coerce"
)

df["net_inflow_crore"] = pd.to_numeric(
    df["net_inflow_crore"],
    errors="coerce"
)

df = df.dropna(
    subset=["category"]
)

df.to_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\processed\\05_category_inflows_clean.csv", index=False)

### industry no 6


In [7]:
import pandas as pd
df=pd.read_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\raw\\06_industry_folio_count.csv")
folio_cols = [
    "total_folios_crore",
    "equity_folios_crore",
    "debt_folios_crore",
    "hybrid_folios_crore",
    "others_folios_crore"
]

for col in folio_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df = df[df[col] >= 0]

df.to_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\processed\\06_industry_folio_count_clean.csv", index=False)

## #09 portfolio

In [8]:
import pandas as pd
df= pd.read_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\raw\\09_portfolio_holdings.csv")
df["portfolio_date"] = pd.to_datetime(
    df["portfolio_date"],
    errors="coerce"
)

df["weight_pct"] = pd.to_numeric(
    df["weight_pct"],
    errors="coerce"
)

df = df[
    (df["weight_pct"] >= 0) &
    (df["weight_pct"] <= 100)
]

df = df[
    df["market_value_cr"] > 0
]

df = df[
    df["current_price_inr"] > 0
]
df.to_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\processed\\09_portfolio_holdings.csv", index=False)

C:\Users\Mohit\AppData\Local\Temp\ipykernel_20448\1755300534.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["portfolio_date"] = pd.to_datetime(


### 10 benchmark


In [9]:
import pandas as pd
df = pd.read_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\raw\\10_benchmark_indices.csv")
df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df["close_value"] = pd.to_numeric(
    df["close_value"],
    errors="coerce"
)

df = df[
    df["close_value"] > 0
]
df.to_csv("C:\\Users\\Mohit\\OneDrive\\Desktop\\bluestock_mf_capstone\\Data\\processed\\10_benchmark_indices_clean.csv", index=False)